In [ ]:
import pandas as pd
import glob

# Combina os arquivos csv de 2010 até 2024 (ignoro o ATP Database por enquanto)
arquivos = sorted(glob.glob("../data/raw/20*.csv"))
df = pd.concat([pd.read_csv(f, low_memory=False) for f in arquivos], ignore_index=True)

df.shape

In [ ]:
df.shape
df.info()
df.isna().mean().sort_values(ascending=False).head(15)

In [ ]:
# Agrupa a proporção de ausentes por torneio. Quero saber se a quantidade de nulos é aleatória ou é maior em torneios alternativos
df['stat_ausente'] = df['w_ace'].isna()
df.groupby('tourney_level')['stat_ausente'].mean().sort_values(ascending=False)
# Confirma a hipótese, os torneios alternativos têm mais ausentes

In [ ]:
# Verifica a proporção de cada torneio. Quero analisar a possibilidade de excluir os torneios alternativos da análise
df['tourney_level'].value_counts(normalize=True).round(3)

In [ ]:
# Tira as olimpiadas (O) e a copa davis (D): são competições em equipes com regras diferentes dos torneios que quero analisar
# ATP Finals (F) e outros (A) são menos de 2% do dataset, não vão atrapalhar a análise 
df_filtrado = df[~df['tourney_level'].isin(['D', 'O'])].copy()
print(df.shape, "->", df_filtrado.shape)
df_filtrado['w_ace'].isna().mean()

In [ ]:
df_limpo = df_filtrado.dropna(subset=['w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon', 'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon']).copy()
                                        
df_limpo.shape

O objetivo das próximas três células é transformar as estatísticas do vencedor e do perdedor em linhas distintas, ficando duas linhas por partida. Assim evito ser tendencioso pro lado do vencedor ao analisar desempenho de saque

In [ ]:
colunas_vencedor = ['tourney_id', 'tourney_date', 'surface', 'tourney_level', 'best_of', 'round', 'winner_id', 'winner_name', 'winner_hand', 'winner_ht', 'winner_age', 'winner_rank', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon', 'w_SvGms']

colunas_perdedor = ['tourney_id', 'tourney_date', 'surface', 'tourney_level', 'best_of', 'round', 'loser_id', 'loser_name', 'loser_hand', 'loser_ht', 'loser_age', 'loser_rank', 'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms']

nomes_novos = ['tourney_id', 'tourney_date', 'surface', 'tourney_level', 'best_of', 'round', 'player_id', 'player_name', 'hand', 'ht', 'age', 'rank', 'ace', 'df', 'svpt', 'firstIn', 'firstWon', 'secondWon', 'SvGms']

In [ ]:
venceu = df_limpo[colunas_vencedor].copy()
venceu.columns = nomes_novos
venceu['venceu_partida'] = 1

perdeu = df_limpo[colunas_perdedor].copy()
perdeu.columns = nomes_novos
perdeu['venceu_partida'] = 0

In [ ]:
jogadores = pd.concat([venceu, perdeu], ignore_index=True)
jogadores.shape

In [ ]:
jogadores['pct_pontos_saque_ganhos'] = (jogadores['firstWon'] + jogadores['secondWon']) / jogadores['svpt'] * 100
jogadores['pct_primeiro_saque_dentro'] = jogadores['firstIn'] / jogadores['svpt'] * 100
jogadores['aces_por_100_saques'] = jogadores['ace'] / jogadores['svpt'] * 100

jogadores[['pct_pontos_saque_ganhos', 'pct_primeiro_saque_dentro', 'aces_por_100_saques']].describe()

In [ ]:
(jogadores['svpt'] == 0).sum()

In [ ]:
jogadores_validos = jogadores[jogadores['svpt'] > 0].copy()

inconsistentes = jogadores_validos[jogadores_validos['pct_primeiro_saque_dentro'] > 100]
print("Linhas com % de 1º saque > 100%:", len(inconsistentes))
inconsistentes[['player_name', 'tourney_id', 'firstIn', 'svpt', 'pct_primeiro_saque_dentro']].head(10)

In [ ]:
jogadores_final = jogadores_validos[jogadores_validos['pct_primeiro_saque_dentro'] <= 100].copy()
jogadores_final.shape

In [ ]:
jogadores_final.to_csv("../data/processed/jogadores_saque.csv", index=False)

### Resumo do tratamento
- Exclusão conceitual da Davis Cup e Olimpíadas (equipe)
- Exclusão de 22 partidas sem pontos de saque registrados
- Exclusão de uma partida com inconsistência lógica (>100% primeiro saque)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(8,5))
sns.histplot(jogadores_final['pct_pontos_saque_ganhos'], bins=40, kde=True, ax=ax)
ax.set_title('Distribuição de % de pontos de saque ganhos')
ax.set_xlabel('% pontos de saque ganhos')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sns.boxplot(data=jogadores_final, x='surface', y='pct_pontos_saque_ganhos', ax=ax)
ax.set_title('% de pontos de saque ganhos por superfície')
plt.show()

Os gráficos mostraram que há uma concentração que foge da média para baixo na % de saques ganhos. A próxima célula é para verificar se isso tá relacionado com baixo volume de pontos sacados (devido a lesão ou desistência precoce) ou simplesmente baixo desempenho de atletas.

In [ ]:
outliers_baixos = jogadores_final[jogadores_final['pct_pontos_saque_ganhos'] < 20]
print("Quantidade:", len(outliers_baixos))
outliers_baixos['svpt'].describe()

O primeiro caso se confirmou, definir minimo de pontos sacados resolve o problema

In [ ]:
jogadores_final = jogadores_final[jogadores_final['svpt'] >= 20].copy()
jogadores_final.shape

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sns.histplot(jogadores_final['pct_pontos_saque_ganhos'], bins=40, kde=True, ax=ax)
ax.set_title('Distribuição de % de pontos de saque ganhos (svpt >= 20)')
plt.show()

In [ ]:
jogadores_final.to_csv("../data/processed/jogadores_saque.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sns.regplot(data=jogadores_final, x='ht', y='pct_pontos_saque_ganhos', scatter_kws={'alpha':0.05, 's':10}, line_kws={'color':'red'}, ax=ax)
ax.set_title('Altura vs % de pontos de saque ganhos')
ax.set_xlabel('Altura (cm)')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sns.regplot(data=jogadores_final, x='age', y='pct_pontos_saque_ganhos', scatter_kws={'alpha':0.05, 's':10}, line_kws={'color':'red'}, ax=ax)
ax.set_title('Idade vs % de pontos de saque ganhos')
ax.set_xlabel('Idade')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sns.regplot(data=jogadores_final, x='rank', y='pct_pontos_saque_ganhos', scatter_kws={'alpha':0.05, 's':10}, line_kws={'color':'red'}, ax=ax)
ax.set_xscale('log')
ax.set_title('Ranking (escala log) vs % de pontos de saque ganhos')
ax.set_xlabel('Ranking ATP (escala log)')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
sns.boxplot(data=jogadores_final, x='hand', y='pct_pontos_saque_ganhos', ax=ax)
ax.set_title('% de pontos de saque ganhos por mão dominante')
plt.show()